<a href="https://colab.research.google.com/github/Vishwas-Chaudhary/ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
import duckdb
from google.colab import userdata

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Check Colab Secrets.")

# Connect to DuckDB
con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# Authenticate with Hugging Face
con.execute("""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

# March 2026 partition
table_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

# Verify the grain
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet('{table_path}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

# Verify row count and date range
window_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{table_path}')
""").df()

print("UNIT OF ANALYSIS")
print("One row represents one content item for one client on one reporting date.")

print("\nTIME WINDOW")
print("Development and verification window: March 2026.")
print("June 2026 is treated as the sealed final/outcome month.")

print("\nGRAIN CHECK")
print(grain_check)

print("\nMARCH 2026 WINDOW CHECK")
print(window_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

UNIT OF ANALYSIS
One row represents one content item for one client on one reporting date.

TIME WINDOW
Development and verification window: March 2026.
June 2026 is treated as the sealed final/outcome month.

GRAIN CHECK
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, row_count]
Index: []

MARCH 2026 WINDOW CHECK
   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
print("""
FIELD CONTRACT

FEATURES:
1. gsc_avg_position
2. gsc_impressions
3. gsc_clicks
4. ga4_sessions
5. scroll_events

LABEL / PROXY:
Future CTR performance for the content item, calculated from observed
April 2026 clicks and impressions.

CONTEXT:
- client_hash_id: identifies the client and can be used for grouping.
- content_hash_id: identifies the content item.
- report_date: identifies the reporting date and controls the time window.
- gsc_data_available: indicates whether GSC data is available.
- ga4_data_available: indicates whether GA4 data is available.

EXCLUDED:
- trend_direction and trend_pct: excluded because they contain
  outcome-derived trend information and could cause leakage.
- Future-period performance fields: excluded because they are not
  available at the March decision moment.
- Client and content identifiers: excluded from model features because
  they identify entities rather than describe performance.

The five selected features are available from March 2026 data before
the future April outcome is observed.
""")


FIELD CONTRACT

FEATURES:
1. gsc_avg_position
2. gsc_impressions
3. gsc_clicks
4. ga4_sessions
5. scroll_events

LABEL / PROXY:
Future CTR performance for the content item, calculated from observed
April 2026 clicks and impressions.

CONTEXT:
- client_hash_id: identifies the client and can be used for grouping.
- content_hash_id: identifies the content item.
- report_date: identifies the reporting date and controls the time window.
- gsc_data_available: indicates whether GSC data is available.
- ga4_data_available: indicates whether GA4 data is available.

EXCLUDED:
- trend_direction and trend_pct: excluded because they contain
  outcome-derived trend information and could cause leakage.
- Future-period performance fields: excluded because they are not
  available at the March decision moment.
- Client and content identifiers: excluded from model features because
  they identify entities rather than describe performance.

The five selected features are available from March 2026 data b

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# ============================================================
# SECTION 3: VERIFY IT WITH QUERIES + FEATURES + LEAKAGE TRAP
# ============================================================

# ------------------------------------------------------------
# Query 1: Verify the grain
# One row should represent:
# report_date + client_hash_id + content_hash_id
# ------------------------------------------------------------

grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet('{table_path}')
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("QUERY 1: GRAIN CHECK")
print(grain_check)


# ------------------------------------------------------------
# Query 2: March 2026 row count and date span
# ------------------------------------------------------------

window_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{table_path}')
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

print("\nQUERY 2: MARCH 2026 ROW COUNT AND DATE SPAN")
print(window_check)


# ------------------------------------------------------------
# Query 3: Availability using IS TRUE
# ------------------------------------------------------------

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS available_rows
    FROM read_parquet('{table_path}')
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

print("\nQUERY 3: GSC AVAILABILITY")
print(availability_check)


# ------------------------------------------------------------
# Five-feature frame
# Only use data available during March.
# Both GSC and GA4 availability are required.
# ------------------------------------------------------------

march_features = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_avg_position,
        gsc_impressions,
        gsc_clicks,
        ga4_sessions,
        scroll_events
    FROM read_parquet('{table_path}')
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
""").df()

print("\nFIVE-FEATURE FRAME")
print(march_features.head())

print("\nFeature columns:")
print([
    "gsc_avg_position",
    "gsc_impressions",
    "gsc_clicks",
    "ga4_sessions",
    "scroll_events"
])


# ------------------------------------------------------------
# Available-when explanation
# ------------------------------------------------------------

print("""
AVAILABLE WHEN?

1. gsc_avg_position:
   Available at the decision moment because March search position
   has already been observed.

2. gsc_impressions:
   Available at the decision moment because March search impressions
   have already been observed.

3. gsc_clicks:
   Available at the decision moment because March search clicks
   have already been observed.

4. ga4_sessions:
   Available at the decision moment because March sessions
   have already been recorded.

5. scroll_events:
   Available at the decision moment because March scroll activity
   has already been recorded.

The future April outcome is not used as an ordinary feature.
""")


# ------------------------------------------------------------
# Future outcome
# April 2026 is used as the future outcome window.
# ------------------------------------------------------------

april_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-04/*.parquet"
)

april_outcome = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_clicks,
        SUM(gsc_impressions) AS april_impressions,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS future_ctr
    FROM read_parquet('{april_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()


# ------------------------------------------------------------
# Join March features with April future outcome
# ------------------------------------------------------------

feature_frame = march_features.merge(
    april_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("\nFEATURE FRAME WITH FUTURE OUTCOME")
print(feature_frame.head())

print("\nRows with a future outcome:", len(feature_frame))


# ------------------------------------------------------------
# DELIBERATE LEAKAGE TRAP
# Use the future label itself as a fake feature.
# This is intentionally wrong.
# ------------------------------------------------------------

leaky_frame = feature_frame.copy()

leaky_frame["leaked_feature"] = leaky_frame["future_ctr"]

valid_leak = leaky_frame["future_ctr"].notna()

if valid_leak.sum() > 0:

    leak_error = (
        leaky_frame.loc[valid_leak, "future_ctr"]
        - leaky_frame.loc[valid_leak, "leaked_feature"]
    ).abs().mean()

    print("\nLEAKAGE EXPERIMENT")
    print("Rows used:", valid_leak.sum())
    print("Mean absolute error with leaked feature:", leak_error)

    print("""
The score becomes perfect because the leaked feature directly contains
the future outcome that we are trying to predict.
""")


# ------------------------------------------------------------
# Remove the leakage feature
# ------------------------------------------------------------

honest_features = leaky_frame.drop(
    columns=["leaked_feature"]
)

print("\nHONEST FEATURE FRAME AFTER REMOVING LEAKAGE")
print(honest_features.head())

print("""
LEAKAGE LESSON:

The leaked feature was the future April CTR itself. It is unavailable
at the March decision moment because it is calculated from future
April clicks and impressions.

Its perfect score is therefore not real model performance. It is a
leakage artifact caused by giving the system the answer.

The leaked feature has been removed from the honest feature frame.
The remaining five features are the information available at the
March decision moment.
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 1: GRAIN CHECK
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, row_count]
Index: []

QUERY 2: MARCH 2026 ROW COUNT AND DATE SPAN
   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


QUERY 3: GSC AVAILABILITY
   total_rows  available_rows
0     9841378         3611061


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


FIVE-FEATURE FRAME
  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_65de48885f4ef01b  content_5c80451459c29b4a   
1  2026-03-01  client_65de48885f4ef01b  content_b1f61fc81b28b2d4   
2  2026-03-01  client_65de48885f4ef01b  content_e25ea7297a1dffd3   
3  2026-03-01  client_65de48885f4ef01b  content_6b0149a80607dac3   
4  2026-03-01  client_65de48885f4ef01b  content_62673eea26c31c17   

   gsc_avg_position  gsc_impressions  gsc_clicks  ga4_sessions  scroll_events  
0          5.400000                5           0             1              0  
1          5.666667               39           0             2              0  
2          5.156425              179           0             2              0  
3          7.694444               72           0             1              0  
4          6.167885             3282           1             1              0  

Feature columns:
['gsc_avg_position', 'gsc_impressions', 'gsc_clicks', 'ga4_sessions', 'sc

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


FEATURE FRAME WITH FUTURE OUTCOME
  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_65de48885f4ef01b  content_5c80451459c29b4a   
1  2026-03-01  client_65de48885f4ef01b  content_b1f61fc81b28b2d4   
2  2026-03-01  client_65de48885f4ef01b  content_e25ea7297a1dffd3   
3  2026-03-01  client_65de48885f4ef01b  content_6b0149a80607dac3   
4  2026-03-01  client_65de48885f4ef01b  content_62673eea26c31c17   

   gsc_avg_position  gsc_impressions  gsc_clicks  ga4_sessions  scroll_events  \
0          5.400000                5           0             1              0   
1          5.666667               39           0             2              0   
2          5.156425              179           0             2              0   
3          7.694444               72           0             1              0   
4          6.167885             3282           1             1              0   

   april_clicks  april_impressions  future_ctr  
0           0.0     

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
print("""
DATA LIMITATION

The warehouse is an unbalanced panel. Different clients have different
amounts of historical search and analytics data, so the same calendar
window does not necessarily provide the same amount of usable information
for every client.

Data availability flags therefore need to be checked before comparing
content items across clients.

Another limitation is that the analysis uses observed historical
performance for decision support. It does not establish that a particular
factor causes higher or lower CTR.
""")


DATA LIMITATION

The warehouse is an unbalanced panel. Different clients have different
amounts of historical search and analytics data, so the same calendar
window does not necessarily provide the same amount of usable information
for every client.

Data availability flags therefore need to be checked before comparing
content items across clients.

Another limitation is that the analysis uses observed historical
performance for decision support. It does not establish that a particular
factor causes higher or lower CTR.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.